In [ ]:
import json
import torch
import os
import copy
import torchvision
import random
import numpy as np
import torchvision.transforms as transforms
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from torchvision import models
import torch.nn as nn
from tqdm import tqdm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from torchvision.models import (
    ResNet34_Weights,
    ViT_B_16_Weights
)
import timm

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

SEED=42
 
set_seed(SEED)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] Available device: {device}")

In [ ]:
# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
EPOCHS = 100
BATCH_SIZE = 128
LR = 0.01
WD = 5e-4
MOMENTUM = 0.9



# ============================================================================
# OPTIMIZER & SCHEDULER CONFIGURATION
# ============================================================================

OPTIMIZER_TYPE = "SGD"
SCHEDULER_TYPE = "CosineAnnealingLR"


# ============================================================================
# LOSS FUNCTION CONFIGURATION
# ============================================================================
LOSS_TYPE = "CrossEntropyLoss"
LABEL_SMOOTHING = 0.0



# ============================================================================
# StepLR / ExponentialLR
# ============================================================================
STEP_SIZE = 30
GAMMA = 0.1




# ============================================================================
# MultiStepLR
# ============================================================================
MILESTONES = [10, 15]





# ============================================================================
# CosineAnnealingLR
# ============================================================================
T_MAX = EPOCHS
ETA_MIN = 0.0




# ============================================================================
# CosineAnnealingWarmRestarts
# ============================================================================
T_0 = 10         
T_MULT = 2       






# ============================================================================
# ReduceLROnPlateau
# ============================================================================
FACTOR = 0.1       
PATIENCE = 5      
THRESHOLD = 1e-4   
MIN_LR = 1e-6




# ============================================================================
# DATASET & MODEL CONFIGURATION
# ============================================================================
DATASET_NAME = "Food101"
SPLIT = 0.2
MODEL_NAME = "ResNet34"
IMG_SIZE = 224
TSNE_MAX_SAMPLES = 2000

# ============================================================================
# PRETRAINED MODEL FLAG
# ============================================================================
PRETRAINED_MODEL = True


if DATASET_NAME == "CIFAR10":
    NUM_CLASSES = 10
    
elif DATASET_NAME == "CIFAR100":
    NUM_CLASSES = 100

elif DATASET_NAME == "Food101":
    NUM_CLASSES = 101

elif DATASET_NAME == "Flowers102":
    NUM_CLASSES = 102

else:
    raise ValueError(f"Unsupported dataset: {DATASET_NAME}")


# ============================================================================
# PATHS & DIRECTORIES
# ============================================================================
BASE_DIR = os.path.dirname(os.getcwd())
DATA = os.path.join(BASE_DIR, "data")
ROOT_DIR = os.path.join(
    BASE_DIR,
    "model",
    MODEL_NAME,
    DATASET_NAME,
    f"seed_{SEED}",
    f"split_{SPLIT}"
)
SPLIT_DIR = os.path.join(BASE_DIR, "split", DATASET_NAME, f"seed_{SEED}", f"split_{SPLIT}")

os.makedirs(DATA, exist_ok=True)
os.makedirs(ROOT_DIR, exist_ok=True)

In [ ]:
def create_loss():

    if LOSS_TYPE == "CrossEntropyLoss":
        return nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING
        )

    if LOSS_TYPE == "BCEWithLogitsLoss":
        return nn.BCEWithLogitsLoss()

    if LOSS_TYPE == "NLLLoss":
        return nn.NLLLoss()

    raise ValueError(f"Loss '{LOSS_TYPE}' is not supported.")





def create_optimizer(model):
    if OPTIMIZER_TYPE == "SGD":
        return torch.optim.SGD(
            model.parameters(),
            lr=LR,
            momentum=MOMENTUM,
            weight_decay=WD
        )
    
    if OPTIMIZER_TYPE == "Adam":
        return torch.optim.Adam(
            model.parameters(),
            lr=LR,
            weight_decay=WD
        )
    
    if OPTIMIZER_TYPE == "AdamW":
        return torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WD
        )
    
    raise ValueError(f"Optimizer '{OPTIMIZER_TYPE}' is not supported.")







def create_scheduler(optimizer):

    if SCHEDULER_TYPE == "StepLR":
        return torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=STEP_SIZE,
            gamma=GAMMA
        )

    if SCHEDULER_TYPE == "MultiStepLR":
        return torch.optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=MILESTONES,
            gamma=GAMMA
        )

    if SCHEDULER_TYPE == "CosineAnnealingLR":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=T_MAX,
            eta_min=ETA_MIN
        )

    if SCHEDULER_TYPE == "CosineAnnealingWarmRestarts":
        return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=T_0,          
            T_mult=T_MULT,    
            eta_min=ETA_MIN
        )
    
    if SCHEDULER_TYPE == "ReduceLROnPlateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",         
            factor=FACTOR,
            patience=PATIENCE,
            threshold=THRESHOLD,
            min_lr=MIN_LR
        )

    raise ValueError(f"Scheduler '{SCHEDULER_TYPE}' is not supported.")




In [ ]:
if DATASET_NAME in ["CIFAR10", "CIFAR100"]:
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.Resize(IMG_SIZE),  
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    transform_test = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

else:
    transform_train = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    transform_test = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])


In [ ]:
if DATASET_NAME == "CIFAR10":
    train_dataset_full = torchvision.datasets.CIFAR10(root=DATA, train=True, download=True, transform=transform_train)
    val_dataset_full = torchvision.datasets.CIFAR10(root=DATA, train=True, download=True, transform=transform_test)
    test_dataset = torchvision.datasets.CIFAR10(root=DATA, train=False, download=True, transform=transform_test)

elif DATASET_NAME == "CIFAR100":
    train_dataset_full = torchvision.datasets.CIFAR100(root=DATA, train=True, download=True, transform=transform_train)
    val_dataset_full = torchvision.datasets.CIFAR100(root=DATA, train=True, download=True, transform=transform_test)
    test_dataset = torchvision.datasets.CIFAR100(root=DATA, train=False, download=True, transform=transform_test)

elif DATASET_NAME == "Food101":
    train_dataset_full = torchvision.datasets.Food101(root=DATA, split="train", download=True, transform=transform_train)
    val_dataset_full = torchvision.datasets.Food101(root=DATA, split="train", download=True, transform=transform_test)
    test_dataset = torchvision.datasets.Food101(root=DATA, split="test", download=True, transform=transform_test)

elif DATASET_NAME == "Flowers102":
    train_dataset_full = torchvision.datasets.Flowers102(root=DATA, split="train", download=True, transform=transform_train)
    val_dataset_full = torchvision.datasets.Flowers102(root=DATA, split="val", download=True, transform=transform_test)
    test_dataset = torchvision.datasets.Flowers102(root=DATA, split="test", download=True, transform=transform_test)

else:
    raise ValueError(f"Unsupported dataset: {DATASET_NAME}")


In [ ]:
train_indices = np.load(os.path.join(SPLIT_DIR, "train_indices.npy"))
val_indices = np.load(os.path.join(SPLIT_DIR, "val_indices.npy"))

train_dataset = Subset(train_dataset_full, train_indices)
validation_dataset = Subset(val_dataset_full, val_indices)

NUM_WORKERS = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)


In [ ]:
def build_model(arch_name, num_classes):

    if PRETRAINED_MODEL:
        model_dict = {
            "ResNet34": (models.resnet34, ResNet34_Weights.DEFAULT),
            "ViT_B_16": (models.vit_b_16, ViT_B_16_Weights.DEFAULT)
        }

        # timm models (ViT-Small)
        timm_models = {
            "ViT_Small": "vit_small_patch16_224"
        }

        if arch_name in timm_models:
            model = timm.create_model(timm_models[arch_name], pretrained=True, num_classes=num_classes)
            return model.to(device)

        if arch_name not in model_dict:
            raise ValueError(f"Unsupported architecture: {arch_name}")

        model_fn, weights = model_dict[arch_name]
        model = model_fn(weights=weights)

    else:
        model_dict = {
            "ResNet18": models.resnet18,
            "ResNet50": models.resnet50
        }

        if arch_name not in model_dict:
            raise ValueError(f"Unsupported architecture: {arch_name}")

        model_fn = model_dict[arch_name]
        model = model_fn(weights=None)

    if hasattr(model, "heads"):
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)
    else:
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    return model.to(device)

In [ ]:
model = build_model(arch_name=MODEL_NAME, num_classes=NUM_CLASSES)

print(type(model))
if hasattr(model, "heads"):
    print("Output classes:", model.heads.head.out_features)
else:
    print("Output classes:", model.fc.out_features)

In [ ]:
def train_one_epoch(model, train_loader, optimizer, criterion):
    
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc="Training"):

        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)     
        predicted = torch.argmax(outputs, dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()
       
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc

In [ ]:
def evaluate(model, validation_loader, criterion):
    
    model.eval()
    
    running_loss = 0.0
    correct = 0
    total = 0
    running_top5 = 0.0

    with torch.no_grad():
        for images, labels in tqdm(validation_loader, desc="Validation"):

            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)     

            predicted = torch.argmax(outputs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            top5 = outputs.topk(5, dim=1).indices
            running_top5 += top5.eq(labels.view(-1, 1)).any(dim=1).float().sum().item()
            
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    epoch_top5 = 100. * running_top5 / total   

    return epoch_loss, epoch_acc, epoch_top5


In [ ]:

def train(model, train_loader, validation_loader, criterion, optimizer, scheduler):
        
    best_model_val_acc = 0.0
    best_model_val_loss = float("inf")
    best_epoch = 0
    best_val_top5 = 0.0
    best_model = None

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    print(f"Starting training for {MODEL_NAME} on {DATASET_NAME} for {EPOCHS} epochs:\n")

    for epoch in range(EPOCHS):

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_top5 = evaluate(model, validation_loader, criterion)

        if SCHEDULER_TYPE == "ReduceLROnPlateau":
            scheduler.step(val_loss)
        else:
            scheduler.step()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"{MODEL_NAME} | Epoch [{epoch+1}/{EPOCHS}] | Train Acc: {train_acc:.2f}% | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Top5: {val_top5}\n")


        if (val_acc > best_model_val_acc) or (
            val_acc == best_model_val_acc and  val_loss < best_model_val_loss
        ):
            best_model_val_acc = val_acc
            best_model_val_loss = val_loss
            best_val_top5 = val_top5
            best_epoch = epoch + 1
            best_model = copy.deepcopy(model)

    print(f"[OK] Training completed for {MODEL_NAME} on {DATASET_NAME}")
    print(f"Best validation accuracy: {best_model_val_acc:.2f}% at epoch {best_epoch} | Best Val Top-5: {best_val_top5:.2f}%\n")

    return best_model, best_model_val_acc, best_model_val_loss, train_losses, val_losses, train_accs, val_accs, best_epoch, best_val_top5


In [ ]:
def test(model, test_loader, criterion):
    
    model.eval()
    
    running_loss = 0.0
    correct = 0
    total = 0
    running_top5 = 0.0

    y_true = []
    y_pred = []

    print(f"Starting testing for {MODEL_NAME} on {DATASET_NAME}:\n")

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Testing"):

            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)     
            predicted = torch.argmax(outputs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            top5 = outputs.topk(5, dim=1).indices
            running_top5 += top5.eq(labels.view(-1, 1)).any(dim=1).float().sum().item()

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
       
    test_loss = running_loss / total
    test_acc = 100. * correct / total        
    test_top5 = 100. * running_top5 / total   

    print(f"\n[OK] Test completed for {MODEL_NAME} on {DATASET_NAME} | Test Acc: {test_acc:.2f}% | Test Loss: {test_loss:.4f} | Test Top-5: {test_top5:.2f}%\n")

    return test_loss, test_acc, test_top5, y_true, y_pred

In [ ]:
def transforms_to_config(compose_transform):
    cfg = []
    for tr in compose_transform.transforms:
        if isinstance(tr, T.RandomResizedCrop):
            cfg.append({"name": "RandomResizedCrop", "size": tr.size})
        elif isinstance(tr, T.RandomCrop):
            cfg.append({"name": "RandomCrop", "size": tr.size, "padding": tr.padding})
        elif isinstance(tr, T.RandomHorizontalFlip):
            cfg.append({"name": "RandomHorizontalFlip", "p": tr.p})
        elif isinstance(tr, T.Resize):
            cfg.append({"name": "Resize", "size": tr.size})
        elif isinstance(tr, T.CenterCrop):
            cfg.append({"name": "CenterCrop", "size": tr.size})
        elif isinstance(tr, T.ToTensor):
            cfg.append({"name": "ToTensor"})
        elif isinstance(tr, T.Normalize):
            cfg.append({"name": "Normalize", "mean": list(tr.mean), "std": list(tr.std)})
        else:
            raise ValueError(f"Unsupported transform for saving: {tr}")
        
    return cfg







def save_experiment(model, experiment_name, best_epoch, val_acc, val_loss, test_acc, test_loss, val_top5, test_top5):

    os.makedirs(experiment_name, exist_ok=True)

    torch.save(model.state_dict(),
               os.path.join(experiment_name, "model.pth"))

    train_transforms_cfg = transforms_to_config(transform_train)
    test_transforms_cfg = transforms_to_config(transform_test)

    # =========================
    # OPTIMIZER
    # =========================
    optimizer_params = {}
    if OPTIMIZER_TYPE == "SGD":
        optimizer_params = {
            "lr": LR,
            "momentum": MOMENTUM,
            "weight_decay": WD
        }
    elif OPTIMIZER_TYPE in ["Adam", "AdamW"]:
        optimizer_params = {
            "lr": LR,
            "weight_decay": WD
        }

    # =========================
    # SCHEDULER
    # =========================
    scheduler_params = {}
    if SCHEDULER_TYPE == "StepLR":
        scheduler_params = {"step_size": STEP_SIZE, "gamma": GAMMA}
    elif SCHEDULER_TYPE == "MultiStepLR":
        scheduler_params = {"milestones": MILESTONES, "gamma": GAMMA}
    elif SCHEDULER_TYPE == "CosineAnnealingLR":
        scheduler_params = {"t_max": T_MAX, "eta_min": ETA_MIN}
    elif SCHEDULER_TYPE == "CosineAnnealingWarmRestarts":
        scheduler_params = {"t_0": T_0, "t_mult": T_MULT, "eta_min": ETA_MIN}
    elif SCHEDULER_TYPE == "ReduceLROnPlateau":
        scheduler_params = {
            "factor": FACTOR,
            "patience": PATIENCE,
            "threshold": THRESHOLD,
            "min_lr": MIN_LR,
            "mode": "min"
        }

    # =========================
    # LOSS FUNCTION
    # =========================
    loss_params = {}

    if LOSS_TYPE == "CrossEntropyLoss":
        loss_params = {
            "label_smoothing": LABEL_SMOOTHING
        }

    metrics = {
        "model_name": MODEL_NAME,
        "dataset_name": DATASET_NAME,
        "epochs": EPOCHS,
        "best_epoch": best_epoch,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "split": SPLIT,

        "val_acc": val_acc,
        "val_loss": val_loss,
        "test_acc": test_acc,
        "test_loss": test_loss,
        "val_top5": val_top5,
        "test_top5": test_top5,

        "optimizer": {
            "type": OPTIMIZER_TYPE,
            "params": optimizer_params
        },

        "scheduler": {
            "type": SCHEDULER_TYPE,
            "params": scheduler_params
        },

        "loss_function": {
            "type": LOSS_TYPE,
            "params": loss_params
        },

        "transforms_train": train_transforms_cfg,
        "transforms_test": test_transforms_cfg
    }

    with open(os.path.join(experiment_name, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"[OK] Model saved as: model.pth in {experiment_name}\n")
    print(f"[OK] Metrics saved as: metrics.json in {experiment_name}\n")

    return experiment_name








def save_plot_training_curves(run_dir, train_losses, val_losses, train_accs, val_accs):
    
    epochs = range(1, len(train_losses) + 1)

    plt.figure(figsize=(8, 6))
    plt.plot(epochs, train_accs, label="Train Accuracy", linewidth=2)
    plt.plot(epochs, val_accs, label="Validation Accuracy", linewidth=2)
    plt.title(f"{MODEL_NAME} - {DATASET_NAME}", fontsize=14)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Accuracy (%)", fontsize=12)
    plt.legend(fontsize=10)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "plot_accuracy.pdf"), dpi=300)
    plt.close()
    
    plt.figure(figsize=(8, 6))
    plt.plot(epochs, train_losses, label="Train Loss", linewidth=2)
    plt.plot(epochs, val_losses, label="Validation Loss", linewidth=2)
    plt.title(f"{MODEL_NAME} - {DATASET_NAME}", fontsize=14)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.legend(fontsize=10)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "plot_loss.pdf"), dpi=300)
    plt.close()
    
    print(f"[OK] Plots saved as: plot_accuracy.pdf and plot_loss.pdf in {run_dir}\n")










def save_confusion_matrix(run_dir, y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred)
    cmn = cm / np.maximum(1, cm.sum(axis=1, keepdims=True))  
    perc = cmn * 100.0
    perc_r = np.round(perc, 1)                             

   
    for i in range(perc_r.shape[0]):
        diff = 100.0 - perc_r[i].sum()
        if abs(diff) >= 1e-9:
            j = int(np.argmax(perc[i]))                   
            perc_r[i, j] += diff                           

    annot = np.vectorize(lambda x: f"{x:.1f}%")(perc_r)

    plt.figure(figsize=(8, 6))
    sns.heatmap(perc_r, annot=annot, fmt="", cmap="Blues", cbar=False, vmin=0, vmax=100)
    plt.title(f"{MODEL_NAME} - {DATASET_NAME}", fontsize=14)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "cm_test.pdf"), dpi=300)
    plt.close()

    print(f"[OK] Confusion matrix saved as: cm_test.pdf in {run_dir}\n")











def save_per_class_accuracy_sorted(run_dir, y_true, y_pred, k, mode):

    C = int(NUM_CLASSES)

    
    cm = confusion_matrix(y_true, y_pred, labels=range(C)).astype(float)
    support = cm.sum(axis=1)
    acc = np.where(support > 0, np.diag(cm) / support, 0.0) * 100.0

    idx_valid = np.where(support > 0)[0]

    if len(idx_valid) == 0:

        print("[WARN] No classes with support>0; nothing to plot")
        return

    k = int(min(k, len(idx_valid)))
    acc_valid = acc[idx_valid]

    if mode == "bottom":

        sel_in_valid = np.argsort(acc_valid)[:k]
        title = f"{MODEL_NAME} - {DATASET_NAME}"
        fname = f"per_class_test_bottom{k}.pdf"

    else:

        sel_in_valid = np.argsort(-acc_valid)[:k]
        title = f"{MODEL_NAME} - {DATASET_NAME}"
        fname = f"per_class_test_top{k}.pdf"

    order = idx_valid[sel_in_valid]     
    labels = [str(i) for i in order]
    x = np.arange(k)

    fig = plt.figure(figsize=(max(10, k * 0.6), 4))
    ax = plt.gca()
    ax.bar(x, acc[order])
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)   
    ax.set_yticks(np.arange(0, 101, 10))      
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(title)
    ax.set_ylim(0, 100)

    fig.tight_layout()
    fig.savefig(os.path.join(run_dir, fname), dpi=300)
    plt.close(fig)

    print(f"[OK] Per-class {mode} accuracy plot saved as: {fname} in {run_dir}\n")






def save_confusion_matrix_sparse(run_dir, y_true, y_pred, step):

    C = len(set(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=range(C)).astype(float)
    cmn = (cm / np.maximum(1, cm.sum(axis=1, keepdims=True))) * 100.0

    plt.figure(figsize=(10, 10))
    im = plt.imshow(np.clip(cmn, 0, 100), interpolation="nearest", aspect="auto", cmap="Blues", vmin=0, vmax=100)
    cb = plt.colorbar(im)
    cb.set_label("%")

    ticks = np.arange(0, C, step)
    tick_labels = [str(i) for i in ticks]
    plt.xticks(ticks, tick_labels, fontsize=8)
    plt.yticks(ticks, tick_labels, fontsize=8)

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{MODEL_NAME} - {DATASET_NAME}")
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "cm_test.pdf"), dpi=300)
    plt.close()

    print(f"[OK] Confusion matrix saved as: cm_test.pdf in {run_dir}\n")






def extract_embeddings_fc_input(model, loader, max_samples):
    model.eval()
    feats = []
    labs = []

    def hook_fn(module, inputs, outputs):
        feats.append(inputs[0].detach().cpu().flatten(1))

    if hasattr(model, "heads"):
        classifier = model.heads.head
    elif hasattr(model, "head"):
        classifier = model.head
    else:
        classifier = model.fc
    handle = classifier.register_forward_hook(hook_fn)

    seen = 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting embeddings"):
            images = images.to(device)
            labels = labels.to(device)

            _ = model(images)

            labs.append(labels.detach().cpu())
            seen += labels.size(0)

            if seen >= max_samples:
                break

    handle.remove()

    X = torch.cat(feats, dim=0)[:max_samples].numpy()
    y = torch.cat(labs, dim=0)[:max_samples].numpy()

    return X, y







def save_tsne_plot(run_dir, Z, y):

    if NUM_CLASSES <= 10:
        cmap = "tab10"
    else:
        cmap = "gist_ncar"

    plt.figure(figsize=(8, 6))
    plt.scatter(Z[:, 0], Z[:, 1], c=y, s=8, cmap=cmap, alpha=0.8)
    plt.title(f"{MODEL_NAME} - {DATASET_NAME}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "tsne_test.pdf"), dpi=300)
    plt.close()

    print(f"\n[OK] t-SNE plot saved as: tsne_test.pdf in {run_dir}\n")


In [ ]:
criterion = create_loss()
optimizer = create_optimizer(model)
scheduler = create_scheduler(optimizer)

best_model, best_val_acc, best_val_loss, train_losses, val_losses, train_accs, val_accs, best_epoch, best_val_top5 = train(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler
)

test_loss, test_acc, test_top5, y_true_test, y_pred_test = test(
    model=best_model,
    test_loader=test_loader,
    criterion=criterion
)

experiment_name = os.path.join(ROOT_DIR, f"val_acc={best_val_acc:.2f}%_val_loss={best_val_loss:.4f}_lr={LR}_wd={WD}_ep={EPOCHS}_bs={BATCH_SIZE}_opt={OPTIMIZER_TYPE}_sch={SCHEDULER_TYPE}")

run_dir = save_experiment(
    model=best_model,
    experiment_name=experiment_name,
    best_epoch=best_epoch,
    val_acc=best_val_acc,
    val_loss=best_val_loss,
    test_acc=test_acc,
    test_loss=test_loss,
    val_top5=best_val_top5,
    test_top5=test_top5
)

save_plot_training_curves(run_dir=run_dir, train_losses=train_losses, val_losses=val_losses, train_accs=train_accs, val_accs=val_accs)

if DATASET_NAME == "CIFAR10":
    save_confusion_matrix(run_dir=run_dir, y_true=y_true_test, y_pred=y_pred_test)
else:
    save_per_class_accuracy_sorted(run_dir=run_dir, y_true=y_true_test, y_pred=y_pred_test, k=20, mode="bottom")
    save_per_class_accuracy_sorted(run_dir=run_dir, y_true=y_true_test, y_pred=y_pred_test, k=20, mode="top")
    save_confusion_matrix_sparse(run_dir=run_dir, y_true=y_true_test, y_pred=y_pred_test, step=10)

tsne_indices_path = os.path.join(run_dir, "tsne_test_indices.npy")
rng = np.random.RandomState(SEED)
total = len(test_dataset)
k = min(TSNE_MAX_SAMPLES, total)
tsne_indices = rng.choice(total, size=k, replace=False)
np.save(tsne_indices_path, tsne_indices)

tsne_dataset = Subset(test_dataset, tsne_indices)
tsne_loader = DataLoader(tsne_dataset, batch_size=BATCH_SIZE, shuffle=False)

X, y = extract_embeddings_fc_input(model=best_model, loader=tsne_loader, max_samples=len(tsne_indices))
X_scaled = StandardScaler().fit_transform(X)

tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=SEED)
Z = tsne.fit_transform(X_scaled)

save_tsne_plot(run_dir=run_dir, Z=Z, y=y)
